# ⚽ Curso de Big Data e IA — Módulo 1
## Bloques 1-2: Fundamentos de Big Data y tratamiento de datos con Python

**Duración estimada:** 5 horas · **Herramienta:** Google Colab

**Caso práctico:** la temporada **2015/2016 de La Liga del Real Sporting de Gijón** (38 partidos), con datos de eventos
de cada partido: pases, tiros, duelos, recuperaciones... con su posición en el campo.

**Datos:** [StatsBomb Open Data](https://github.com/statsbomb/open-data). Si publicas o compartes análisis basados en estos datos, indica StatsBomb como fuente y usa su logo (disponible en su *media pack*).

**Objetivos**
- Entender las "V" del Big Data con un caso real.
- Leer datos **semiestructurados** (JSON anidado) y convertirlos en tablas con pandas.
- Detectar y corregir problemas de calidad: nulos, tipos y duplicados.
- Hacer un análisis exploratorio con estadísticas y primeros gráficos.
- Dejar dos ficheros limpios (`sporting_partidos.csv` y `sporting_eventos.csv`) que reutilizaremos en el resto del curso.

## 1. Las "V" del Big Data en el fútbol

| V | En nuestro caso |
|---|---|
| **Volumen** | Un partido genera miles de eventos; una temporada de un equipo, más de 100.000 |
| **Velocidad** | Los eventos se registran durante el partido, casi en tiempo real |
| **Variedad** | JSON anidado: cada tipo de evento (pase, tiro, duelo) tiene campos distintos |
| **Veracidad** | Hay campos vacíos por diseño y convenciones que hay que conocer (lo veremos con los pases) |

**Flujo del proyecto:** `INGESTA → ALMACENAMIENTO → PROCESAMIENTO → VISUALIZACIÓN`.
Hoy hacemos la ingesta y el procesamiento con **pandas**.

In [ ]:
import json
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle

pd.set_option("display.max_columns", None)

# Repositorio público de StatsBomb. Si tienes el repositorio clonado en local,
# puedes poner aquí la ruta de la carpeta "data" en lugar de la URL.
BASE = "https://raw.githubusercontent.com/statsbomb/open-data/master/data"
EQUIPO = "Sporting Gijón"

def leer_json(ruta):
    """Lee un JSON desde una URL o desde una carpeta local."""
    if BASE.startswith("http"):
        r = requests.get(f"{BASE}/{ruta}", timeout=60)
        r.raise_for_status()
        return r.json()
    with open(f"{BASE}/{ruta}", encoding="utf-8") as f:
        return json.load(f)

print("Listo ✅")

## 2. Ingesta (I): ¿qué competiciones hay?

El repositorio tiene un catálogo, `competitions.json`. Buscamos La Liga y la temporada 2015/2016.

In [ ]:
competiciones = pd.json_normalize(leer_json("competitions.json"))
print(f"{len(competiciones)} combinaciones competición-temporada")

laliga = competiciones[competiciones["competition_name"] == "La Liga"]
laliga[["competition_id", "season_id", "season_name"]].sort_values("season_name")

La Liga 2015/2016 es la competición **11**, temporada **27**. Esos dos números nos dicen qué fichero de partidos leer.
Es una temporada completa (380 partidos) en la que jugó el Sporting.

## 3. Ingesta (II): los partidos

El JSON tiene objetos anidados (equipos, árbitro, estadio). `json_normalize` los aplana en columnas.

In [ ]:
partidos_raw = pd.json_normalize(leer_json("matches/11/27.json"))
print(partidos_raw.shape)
partidos_raw.head(3)

In [ ]:
# Nos quedamos con los partidos del Sporting (como local o como visitante)
sporting = partidos_raw[
    (partidos_raw["home_team.home_team_name"] == EQUIPO) |
    (partidos_raw["away_team.away_team_name"] == EQUIPO)
].copy()
print(f"Partidos del {EQUIPO}: {len(sporting)}")

## 4. Calidad del dato en los partidos

Antes de limpiar, buscamos problemas: nulos, tipos y duplicados.

In [ ]:
# Columnas con valores nulos
nulos = sporting.isnull().sum()
nulos[nulos > 0]

In [ ]:
print("Duplicados por match_id:", sporting["match_id"].duplicated().sum())
sporting[["match_id", "match_date", "kick_off", "home_score", "away_score"]].dtypes

**Qué vemos:**
- Algunas columnas están **completamente vacías** (grupos de equipo) y no aportan nada.
- Hay **un partido sin árbitro** registrado: dato real incompleto.
- `match_date` es texto, no fecha.
- No hay duplicados.

## 5. Limpieza y transformación de los partidos

Nos quedamos con lo útil, ponemos los tipos correctos y creamos columnas desde el punto de vista del Sporting.

In [ ]:
es_local = sporting["home_team.home_team_name"] == EQUIPO

partidos = pd.DataFrame({
    "match_id": sporting["match_id"],
    "fecha": pd.to_datetime(sporting["match_date"]),
    "jornada": sporting["match_week"],
    "local": es_local,
    "rival": np.where(es_local, sporting["away_team.away_team_name"], sporting["home_team.home_team_name"]),
    "goles_favor": np.where(es_local, sporting["home_score"], sporting["away_score"]),
    "goles_contra": np.where(es_local, sporting["away_score"], sporting["home_score"]),
    "estadio": sporting["stadium.name"],
    "arbitro": sporting["referee.name"].fillna("Desconocido"),
})

partidos["resultado"] = np.select(
    [partidos["goles_favor"] > partidos["goles_contra"],
     partidos["goles_favor"] == partidos["goles_contra"]],
    ["V", "E"], default="D")
partidos["puntos"] = partidos["resultado"].map({"V": 3, "E": 1, "D": 0})

partidos = partidos.sort_values("fecha").reset_index(drop=True)
partidos.head(10)

In [ ]:
print("Nulos restantes:", partidos.isnull().sum().sum())
print(partidos["resultado"].value_counts())
print(f"Puntos totales: {partidos['puntos'].sum()}  |  Goles a favor: {partidos['goles_favor'].sum()}  |  Goles en contra: {partidos['goles_contra'].sum()}")

## 6. Ingesta (III): los eventos de cada partido

Aquí está el **volumen**. Cada partido tiene su fichero de eventos, con miles de registros. Un evento es un pase, un tiro, un duelo...
y **cada tipo tiene campos distintos** (un pase tiene destinatario, un tiro tiene xG). Eso es la **variedad**.

En vez de aplanarlo todo, extraemos los campos que nos interesan con una función. Así controlamos qué sacamos de cada estructura anidada.

> ⏱️ Descargar 38 ficheros puede tardar uno o dos minutos.

In [ ]:
def aplanar_evento(e, match_id):
    """Convierte un evento anidado en una fila plana."""
    fila = {
        "match_id": match_id,
        "event_id": e["id"],
        "index": e["index"],
        "period": e["period"],
        "minute": e["minute"],
        "second": e["second"],
        "type": e["type"]["name"],
        "team": e["team"]["name"],
        "possession": e["possession"],
        "play_pattern": e["play_pattern"]["name"],
        "player": e.get("player", {}).get("name"),
        "position": e.get("position", {}).get("name"),
        "x": e["location"][0] if "location" in e else None,
        "y": e["location"][1] if "location" in e else None,
        "under_pressure": e.get("under_pressure", False),
        "duration": e.get("duration"),
    }
    pase = e.get("pass")
    if pase:
        fila.update({
            "pass_recipient": pase.get("recipient", {}).get("name"),
            "pass_length": pase.get("length"),
            "pass_height": pase.get("height", {}).get("name"),
            "pass_end_x": pase["end_location"][0],
            "pass_end_y": pase["end_location"][1],
            "pass_outcome": pase.get("outcome", {}).get("name"),
        })
    tiro = e.get("shot")
    if tiro:
        fila.update({
            "shot_xg": tiro.get("statsbomb_xg"),
            "shot_outcome": tiro["outcome"]["name"],
            "shot_body_part": tiro["body_part"]["name"],
            "shot_type": tiro["type"]["name"],
            "shot_technique": tiro["technique"]["name"],
            "shot_first_time": tiro.get("first_time", False),
        })
    return fila

In [ ]:
filas = []
for i, mid in enumerate(partidos["match_id"], start=1):
    for e in leer_json(f"events/{mid}.json"):
        filas.append(aplanar_evento(e, mid))
    if i % 10 == 0:
        print(f"{i}/{len(partidos)} partidos leídos")

eventos = pd.DataFrame(filas)
print(f"\nEventos: {eventos.shape[0]:,} filas x {eventos.shape[1]} columnas")
eventos.head()

## 7. Calidad del dato en los eventos

In [ ]:
# % de nulos por columna
(eventos.isnull().mean() * 100).round(1).sort_values(ascending=False).head(15)

**¿Son errores?** No. Los campos `pass_*` solo existen en los eventos de tipo `Pass`, y los `shot_*` solo en los `Shot`.
Son nulos **estructurales**: no faltan, simplemente no aplican.

Pero hay un caso especial que sí hay que conocer: **`pass_outcome`**.

In [ ]:
pases = eventos[eventos["type"] == "Pass"]
print(f"Pases: {len(pases):,}")
print(f"Pases con pass_outcome nulo: {pases['pass_outcome'].isnull().sum():,}")
pases["pass_outcome"].value_counts(dropna=False)

**Convención de los datos:** en un pase, `pass_outcome` **vacío significa que el pase fue completo**. Solo se rellena cuando algo sale mal
(`Incomplete`, `Out`, `Pass Offside`...). Si no lo sabemos y borramos los nulos, **eliminaríamos todos los pases buenos**.

Esto es **veracidad** del dato: entender el significado de un nulo antes de decidir qué hacer con él.

In [ ]:
print("Eventos duplicados (event_id):", eventos["event_id"].duplicated().sum())
eventos.dtypes

## 8. Limpieza y transformación de los eventos

In [ ]:
eventos_c = eventos.copy()

# 1. Pase sin resultado = pase completo
es_pase = eventos_c["type"] == "Pass"
eventos_c.loc[es_pase & eventos_c["pass_outcome"].isnull(), "pass_outcome"] = "Completo"

# 2. En los tiros, first_time vacío significa que no fue de primeras
es_tiro = eventos_c["type"] == "Shot"
eventos_c.loc[es_tiro, "shot_first_time"] = eventos_c.loc[es_tiro, "shot_first_time"].fillna(False)

# 3. Columnas auxiliares útiles para el análisis
eventos_c["es_sporting"] = eventos_c["team"] == EQUIPO
eventos_c["pase_completo"] = np.where(es_pase, eventos_c["pass_outcome"] == "Completo", np.nan)

eventos = eventos_c
print("Pases completos (%):", round(eventos.loc[es_pase, "pase_completo"].mean() * 100, 1))
eventos.loc[es_pase, "pass_outcome"].value_counts()

## 9. Análisis exploratorio

Empezamos por estadísticas descriptivas y después los primeros gráficos.

In [ ]:
tiros = eventos[eventos["type"] == "Shot"]
print(f"Tiros en los 38 partidos (ambos equipos): {len(tiros)}")
tiros[["shot_xg", "x", "y"]].describe()

In [ ]:
# Estadísticas por equipo: pases y porcentaje de acierto
pases = eventos[eventos["type"] == "Pass"]
resumen = (pases.groupby("team")["pase_completo"]
           .agg(pases="size", acierto="mean")
           .sort_values("pases", ascending=False))
resumen["acierto"] = (resumen["acierto"] * 100).round(1)
resumen.head(8)

**Ojo con la comparación:** los rivales solo aparecen en los 2 partidos que jugaron contra el Sporting, mientras que el Sporting aparece en los 38. Por eso comparar el **número** de pases entre equipos no tiene sentido, pero sí el **porcentaje** de acierto.

### 9.1 Primeros gráficos

Tres gráficos básicos: **histograma** (una variable numérica), **barras** (una categórica) y **boxplot** (numérica según grupo).

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(18, 4.5))

pases["pass_length"].hist(bins=40, ax=axs[0], color="#3FB8AF")
axs[0].set_title("Longitud de los pases (metros)")

eventos["type"].value_counts().head(8).plot(kind="bar", ax=axs[1], color="#F2A541")
axs[1].set_title("Tipos de evento más frecuentes")

tiros.boxplot("shot_xg", by="shot_outcome", ax=axs[2], rot=45)
axs[2].set_title("xG según resultado del tiro")
plt.suptitle("")

plt.tight_layout()
plt.show()

**Para comentar:** ¿qué tipo de evento domina? ¿Los tiros que acaban en gol tienen un xG mayor?
(**xG** = *expected goals*: probabilidad de que un tiro acabe en gol según dónde y cómo se hace).

In [ ]:
# Evolución de los puntos del Sporting durante la temporada
partidos["puntos_acum"] = partidos["puntos"].cumsum()

plt.figure(figsize=(10, 4))
plt.plot(partidos["jornada"], partidos["puntos_acum"], marker="o", color="#17706A")
plt.title(f"Puntos acumulados del {EQUIPO} (La Liga 2015/16)")
plt.xlabel("Jornada")
plt.ylabel("Puntos")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Puntos y goles como local y como visitante
partidos.groupby("local")[["puntos", "goles_favor", "goles_contra"]].sum().rename(index={True: "Local", False: "Visitante"})

### 9.2 Mapa de tiros

Los datos incluyen la posición (`x`, `y`) de cada evento. El campo mide 120 x 80 y el equipo siempre ataca de izquierda a derecha.

In [ ]:
def dibujar_campo(ax):
    """Dibuja un campo de 120x80 con las coordenadas de StatsBomb."""
    ax.set_xlim(-2, 122)
    ax.set_ylim(82, -2)
    ax.add_patch(Rectangle((0, 0), 120, 80, fill=False, ec="#555"))
    ax.plot([60, 60], [0, 80], color="#555", lw=1)
    ax.add_patch(Circle((60, 40), 10, fill=False, ec="#555"))
    for x0, ancho in [(0, 18), (102, 18)]:
        ax.add_patch(Rectangle((x0, 18), ancho, 44, fill=False, ec="#555"))
    for x0, ancho in [(0, 6), (114, 6)]:
        ax.add_patch(Rectangle((x0, 30), ancho, 20, fill=False, ec="#555"))
    ax.set_aspect("equal")
    ax.axis("off")

tiros_sp = tiros[tiros["team"] == EQUIPO]
goles = tiros_sp[tiros_sp["shot_outcome"] == "Goal"]
resto = tiros_sp[tiros_sp["shot_outcome"] != "Goal"]

fig, ax = plt.subplots(figsize=(10, 6.5))
dibujar_campo(ax)
ax.scatter(resto["x"], resto["y"], s=resto["shot_xg"] * 1000 + 20, alpha=0.45, color="#2E86AB", label="Sin gol")
ax.scatter(goles["x"], goles["y"], s=goles["shot_xg"] * 500 + 60, marker="*", color="#D1495B", edgecolor="black", label="Gol")
ax.set_title(f"Tiros del {EQUIPO} (el tamaño indica el xG)")
ax.legend(loc="lower left")
plt.show()
print(f"Tiros: {len(tiros_sp)}  |  Goles: {len(goles)}")

## 10. Guardar los datasets limpios

Guardamos las dos tablas en CSV. Las usaremos en:
- **Databricks / Spark** (Bloque 3): hay que **subirlas manualmente**.
- **Visualización / BI** y **Machine Learning**: se suben al panel de archivos de Colab.

In [ ]:
import os

partidos.to_csv("sporting_partidos.csv", index=False)
eventos.to_csv("sporting_eventos.csv", index=False)

for f in ["sporting_partidos.csv", "sporting_eventos.csv"]:
    print(f"{f}: {os.path.getsize(f) / 1024:,.0f} KB")

# En Colab, para descargarlos a tu ordenador, descomenta:
# from google.colab import files
# files.download("sporting_partidos.csv")
# files.download("sporting_eventos.csv")

## 📝 Ejercicio propuesto

1. Calcula el **porcentaje de pases completos del Sporting** y compáralo con el de sus rivales. ¿Qué dice del estilo de juego?
2. ¿Qué jugador del Sporting hizo más tiros? ¿Y más goles? (pista: `groupby("player")` sobre `tiros_sp`).
3. Elige **dos variables** que creas que influyen en ganar un partido y justifícalo con un gráfico.
4. ¿Por qué sería un error aplicar `dropna()` sobre `pass_outcome`?

> La solución del punto 1 y del resto de métricas se amplía en el notebook de Databricks (Spark).

### Cómo usar estas celdas

Cada ejercicio tiene una celda de código con **varias opciones**. Todas empiezan comentadas (con `#` delante), así que de primeras **no hacen nada**.

1. Lee las opciones y elige **una**.
2. Quita el `#` de delante de las líneas de esa opción (en Colab: selecciona las líneas y pulsa **Ctrl + /**, o **⌘ + /** en Mac).
3. Ejecuta la celda con **Mayús + Intro**.
4. Si falla, vuelve a comentar la opción y prueba otra. Equivocarse aquí no rompe nada.

> Las tablas y variables que necesitas ya existen (`pases`, `tiros`, `tiros_sp`, `goles`, `partidos`, `eventos`). Si ves un error de «no está definido», ejecuta antes todas las celdas anteriores.

### Ejercicio 1 · Pases completos: Sporting frente a rivales

**Pista:** la tabla `pases` tiene la columna `pase_completo` (1 = completo, 0 = no) y la columna `es_sporting` (`True` = pase del Sporting, `False` = pase del rival). La media de una columna de 0 y 1 es el **porcentaje** de unos.

**Funciones que puedes usar:** `groupby()`, `mean()`, `round()`, `plot(kind="bar")`.

In [ ]:
# EJERCICIO 1 · % de pases completos: Sporting vs rivales
# Elige UNA opción y quita el # de las líneas.

# --- OPCIÓN A: agrupar por equipo (la más corta) ---
# resultado = pases.groupby("es_sporting")["pase_completo"].mean() * 100
# resultado = resultado.rename(index={True: "Sporting", False: "Rivales"})
# print(resultado.round(1))

# --- OPCIÓN B: calcular cada equipo por separado ---
# pases_sporting = pases[pases["es_sporting"] == True]
# pases_rivales = pases[pases["es_sporting"] == False]
# print("Sporting:", round(pases_sporting["pase_completo"].mean() * 100, 1), "%")
# print("Rivales :", round(pases_rivales["pase_completo"].mean() * 100, 1), "%")

# --- OPCIÓN C: lo mismo, pero con un gráfico de barras ---
# porcentajes = pases.groupby("es_sporting")["pase_completo"].mean() * 100
# porcentajes.index = ["Rivales", "Sporting"]
# porcentajes.plot(kind="bar", color=["#F2A541", "#17706A"])
# plt.ylabel("% de pases completos")
# plt.ylim(0, 100)
# plt.show()

# Para pensar: ¿qué estilo de juego sugiere la diferencia? (pases cortos y de posesión, o juego más directo)

### Ejercicio 2 · ¿Quién tira más y quién marca más?

**Pista:** `tiros_sp` son los tiros del Sporting y `goles` son solo los tiros que acabaron en gol. Cada fila es un tiro; contar filas por jugador es contar tiros.

**Funciones que puedes usar:** `groupby("player")`, `size()`, `value_counts()`, `sort_values(ascending=False)`, `head(5)`, `agg()`.

In [ ]:
# EJERCICIO 2 · Tiros y goles por jugador
# Elige UNA opción y quita el # de las líneas.

# --- OPCIÓN A: con groupby + size (cuenta filas por jugador) ---
# tiros_jugador = tiros_sp.groupby("player").size().sort_values(ascending=False)
# goles_jugador = goles.groupby("player").size().sort_values(ascending=False)
# print("Más tiros:\n", tiros_jugador.head(5))
# print("Más goles:\n", goles_jugador.head(5))

# --- OPCIÓN B: con value_counts (hace lo mismo en menos líneas) ---
# print("Más tiros:\n", tiros_sp["player"].value_counts().head(5))
# print("Más goles:\n", goles["player"].value_counts().head(5))

# --- OPCIÓN C: una sola tabla con tiros, goles y xG por jugador ---
# tabla = tiros_sp.groupby("player").agg(
#     tiros=("event_id", "size"),
#     goles=("shot_outcome", lambda s: (s == "Goal").sum()),
#     xg=("shot_xg", "sum"),
# ).sort_values("tiros", ascending=False)
# tabla.head(10).round(2)

# --- OPCIÓN D: gráfico de barras de los 8 que más tiran ---
# tiros_sp["player"].value_counts().head(8).sort_values().plot(kind="barh", color="#2E86AB")
# plt.xlabel("Número de tiros")
# plt.show()

# Para pensar: ¿el que más tira es el que más marca? ¿Qué te dice la columna xg de la opción C?

### Ejercicio 3 · ¿Qué variables influyen en ganar?

**Paso 1.** La primera celda construye una tabla con **una fila por partido** y cuatro variables candidatas: `tiros`, `xg`, `pases` y `acierto_pase`. **Ejecútala tal cual, sin cambiar nada.**

**Paso 2.** En la segunda celda elige una variable (cambia el texto de `variable`) y una opción de gráfico. Repite con una segunda variable y compara.

**Funciones que puedes usar:** `groupby("resultado")`, `mean()`, `boxplot()`, `scatter()`, `corr()`.

In [ ]:
# PASO 1 · Tabla con una fila por partido (ejecútala sin cambiar nada)
pases_sp = pases[pases["es_sporting"]]

por_partido = partidos.set_index("match_id")[["resultado", "puntos"]].copy()
por_partido["tiros"] = tiros_sp.groupby("match_id").size()
por_partido["xg"] = tiros_sp.groupby("match_id")["shot_xg"].sum()
por_partido["pases"] = pases_sp.groupby("match_id").size()
por_partido["acierto_pase"] = pases_sp.groupby("match_id")["pase_completo"].mean() * 100
por_partido = por_partido.fillna(0)   # un partido sin tiros cuenta como 0 tiros

por_partido.head()

In [ ]:
# PASO 2 · Elige la variable y una opción
variable = "xg"     # prueba también: "tiros", "pases", "acierto_pase"

# --- OPCIÓN A: valor medio de la variable según el resultado (V = victoria, E = empate, D = derrota) ---
# print(por_partido.groupby("resultado")[variable].mean().round(2))

# --- OPCIÓN B: boxplot (la caja muestra dónde está la mayoría de los partidos) ---
# por_partido.boxplot(variable, by="resultado")
# plt.suptitle("")
# plt.title(f"{variable} según el resultado")
# plt.show()

# --- OPCIÓN C: dispersión (cada punto es un partido) ---
# plt.scatter(por_partido[variable], por_partido["puntos"], alpha=0.6, color="#17706A")
# plt.xlabel(variable)
# plt.ylabel("Puntos del partido (0, 1 o 3)")
# plt.show()

# --- OPCIÓN D: correlación de todas las variables con los puntos (cerca de 0 = no influye; cerca de 1 = influye mucho) ---
# print(por_partido[["puntos", "tiros", "xg", "pases", "acierto_pase"]].corr()["puntos"].round(2))

# Para pensar: ¿se ve diferencia entre ganar y perder? ¿Es causa o solo coincidencia? (solo hay 38 partidos)

### Ejercicio 4 · ¿Por qué `dropna()` sería un error en `pass_outcome`?

**Pista:** recuerda la convención de la sección 7. En el notebook ya hemos rellenado esos nulos con `"Completo"`, así que para ver qué habría pasado **volvemos a poner los nulos** con `replace` y comparamos.

**Funciones que puedes usar:** `replace()`, `dropna()`, `len()`.

In [ ]:
# EJERCICIO 4 · ¿Qué pasaría si borramos los nulos?
# Elige UNA opción y quita el # de las líneas.

# --- OPCIÓN A: pases que quedarían al borrar los nulos de pass_outcome ---
# antes = len(pases)
# sin_nulos = pases["pass_outcome"].replace("Completo", np.nan).dropna()
# print(f"Pases antes: {antes:,}  |  después del dropna: {len(sin_nulos):,}")
# print(f"Perdemos el {(1 - len(sin_nulos) / antes) * 100:.0f}% de los pases, y todos los que perdemos son los buenos")

# --- OPCIÓN B: ¿qué pasa si hacemos dropna() sobre TODA la tabla de eventos? ---
# print("Filas antes:", len(eventos))
# print("Filas después de dropna():", len(eventos.dropna()))

# --- OPCIÓN C: el porcentaje de acierto antes y después del error ---
# bien = pases["pase_completo"].mean() * 100
# mal = pases["pass_outcome"].replace("Completo", np.nan).dropna().eq("Completo").mean() * 100
# print(f"Acierto correcto: {bien:.1f}%  |  acierto tras el dropna: {mal:.1f}%")

# Para pensar: ¿qué otra forma tenemos de tratar un nulo, aparte de borrarlo?